In [1]:
import pandas as pd
import numpy as np
import pickle
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

# -----------------------------
# LOAD DATA
# -----------------------------
df = pd.read_csv("spotify.csv")
df = df.dropna()

df.rename(columns={"track_name": "song_name"}, inplace=True)

# Drop unwanted column if exists
if "Unnamed: 0" in df.columns:
    df.drop(columns=["Unnamed: 0"], inplace=True)

features = ["energy", "danceability", "tempo", "valence", "loudness"]

# -----------------------------
# SCALE FEATURES
# -----------------------------
scaler = MinMaxScaler()
df[features] = scaler.fit_transform(df[features])

# -----------------------------
# MOOD LABELING
# -----------------------------
def get_mood(row):
    if row["energy"] > 0.75 and row["valence"] > 0.6:
        return "Energetic"
    elif row["valence"] < 0.3:
        return "Sad"
    elif row["energy"] < 0.4:
        return "Calm"
    elif row["danceability"] > 0.7:
        return "Party"
    else:
        return "Happy"

df["mood"] = df.apply(get_mood, axis=1)

# -----------------------------
# TRAIN-TEST SPLIT (IMPORTANT)
# -----------------------------
X = df[features]
y_mood = df["mood"]
y_pop = df["popularity"]

X_train, X_test, y_train_mood, y_test_mood = train_test_split(
    X, y_mood, test_size=0.2, random_state=42
)

X_train_p, X_test_p, y_train_pop, y_test_pop = train_test_split(
    X, y_pop, test_size=0.2, random_state=42
)

# -----------------------------
# MOOD MODEL
# -----------------------------
mood_model = RandomForestClassifier(n_estimators=100, random_state=42)
mood_model.fit(X_train, y_train_mood)

# -----------------------------
# POPULARITY MODEL
# -----------------------------
pop_model = RandomForestRegressor(n_estimators=100, random_state=42)
pop_model.fit(X_train_p, y_train_pop)

# -----------------------------
# RECOMMENDER (KNN)
# -----------------------------
recommender = NearestNeighbors(n_neighbors=6, metric="cosine")
recommender.fit(df[features])

# -----------------------------
# 📊 GRAPH (ONLY ONE - REQUIRED)
# -----------------------------
plt.figure()
plt.scatter(df["energy"], df["valence"])
plt.xlabel("Energy")
plt.ylabel("Valence")
plt.title("Energy vs Valence Distribution")
plt.savefig("graph.png")
plt.close()

# -----------------------------
# 📊 CONFUSION MATRIX
# -----------------------------
y_pred = mood_model.predict(X_test)

cm = confusion_matrix(y_test_mood, y_pred)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=mood_model.classes_
)

disp.plot()
plt.title("Confusion Matrix - Mood Classification")
plt.savefig("confusion.png")
plt.close()

# -----------------------------
# SAVE MODELS
# -----------------------------
pickle.dump(pop_model, open("popularity_model.pkl", "wb"))
pickle.dump(mood_model, open("mood_model.pkl", "wb"))
pickle.dump(scaler, open("scaler.pkl", "wb"))
pickle.dump(df, open("songs.pkl", "wb"))
pickle.dump(recommender, open("recommender.pkl", "wb"))

print("✅ Models + Graph + Confusion Matrix saved successfully!")


✅ Models + Graph + Confusion Matrix saved successfully!
